In [2]:
import pandas as pd

mapa_uf = {"ACRE": "AC", "AMAPÁ": "AP", "AMAZONAS": "AM", "MARANHÃO": "MA",
           "MATO GROSSO": "MT", "PARÁ": "PA", "RONDÔNIA": "RO",
           "RORAIMA": "RR", "TOCANTINS": "TO"}

sem = pd.read_csv("dataset_semanal_COMPLETO.csv", parse_dates=["semana"])
sem = sem.loc[sem["semana"] >= "2024-07-01",
              ["semana", "municipio_norm", "uf", "riesgo_fuego_promedio"]]

sis = pd.read_csv("sisam_limpio.csv", parse_dates=["data"])
sis = sis[sis["data"] >= "2024-07-01"].copy()
sis["uf"] = sis["estado"].map(mapa_uf)
print("Estados sin mapear:", sis["uf"].isna().sum())          # debe dar 0

sis["semana"] = sis["data"].dt.to_period("W").dt.start_time
sis["sobre_oms"] = sis["pm2_5_reanalise"] > 15                 # guía OMS 2021 (24 h)
sis["sobre_nacional"] = sis["pm2_5_reanalise"] > 60            # PI-1, vigente jul-dic 2024

pm = (sis.groupby(["semana", "municipio_norm", "uf"])
         .agg(dias_con_dato=("data", "nunique"),
              pm25_dias_sobre_oms=("sobre_oms", "sum"),
              pm25_dias_sobre_nacional=("sobre_nacional", "sum"))
         .reset_index())

ctx = sem.merge(pm, on=["semana", "municipio_norm", "uf"], how="left")
print(ctx.shape)                                    # debe dar (2726, 7)
print("Sin dato de PM2.5:", ctx["pm25_dias_sobre_oms"].isna().sum())   # debe dar 0
print(ctx["dias_con_dato"].value_counts())          # casi todo 7; la semana parcial del 30-dic da 2
ctx.to_csv("contexto_semanal.csv", index=False)

Estados sin mapear: 0
(2726, 7)
Sin dato de PM2.5: 0
dias_con_dato
7    2724
2       2
Name: count, dtype: int64
